# EEGNet on Colab GPU

Run the EEGNet batch (14 subjects × 5-fold CV × 40 epochs) on a Colab T4 GPU instead of CPU. ~10–20 min total.

## One-time setup before running this notebook

1. **Enable GPU**: `Runtime` → `Change runtime type` → `T4 GPU` (or any GPU) → Save.
2. **Upload the preprocessed .fif files to Google Drive**:
   - In Drive, create a folder `eeg-speech-data/processed/`
   - Drag the 14 `MM*-clean-epo.fif` + `P02-clean-epo.fif` files (and the matching `*-log.json` files) from your local `/Volumes/Josh G/eeg-speech-data/processed/` into that Drive folder
   - Total ~1.7 GB. Wait for it to fully sync before continuing.
3. Run all cells below.

In [ ]:
# 1. Confirm GPU is attached
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime → Change runtime type → GPU, then Run all again.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'torch {torch.__version__}, CUDA {torch.version.cuda}')

In [ ]:
# 2. Mount Drive (auth prompt)
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/eeg-speech-data/processed'
OUT_DIR  = '/content/drive/MyDrive/eeg-speech-data/outputs'

import os
fifs = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.fif'))
assert len(fifs) == 14, f'Expected 14 .fif files in {DATA_DIR}, found {len(fifs)}: {fifs}'
print(f'Found {len(fifs)} preprocessed .fif files in Drive ✅')

In [ ]:
# 3. Clone the repo and cd in
%cd /content
![ -d eeg-speech ] || git clone https://github.com/joshegreenfield2/eeg-speech.git
%cd /content/eeg-speech
!git pull --quiet

In [ ]:
# 4. Install the EEG-specific deps that aren't pre-installed on Colab.
#    (torch, numpy, scipy, sklearn, pandas, matplotlib, h5py, PyWavelets are already there.)
!pip install -q mne mne-icalabel autoreject pyprep braindecode==0.8.1 mat73

In [ ]:
# 5. Symlink data/processed → Drive folder (matches the script's expected layout)
import os
os.makedirs('data', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)
if os.path.islink('data/processed') or os.path.exists('data/processed'):
    os.remove('data/processed') if os.path.islink('data/processed') else None
os.symlink(DATA_DIR, 'data/processed')

!ls data/processed/*.fif | wc -l   # should print 14

In [ ]:
# 6. Run EEGNet on all 14 subjects. Auto-detects CUDA via the device patch in src/train.py.
!python -u scripts/run_dl.py --model eegnet --epochs-max 40

In [ ]:
# 7. Copy results back to Drive so they survive the Colab session
import shutil, glob, os
os.makedirs(OUT_DIR, exist_ok=True)

# Each batch run lives at outputs/results/phase4_dl_<timestamp>/
phase4_dirs = sorted(glob.glob('outputs/results/phase4_dl_*'))
if not phase4_dirs:
    raise SystemExit('No phase4 results found — did the run finish?')
latest = phase4_dirs[-1]
dest = os.path.join(OUT_DIR, 'results', os.path.basename(latest))
shutil.copytree(latest, dest, dirs_exist_ok=True)
print(f'Copied {latest} → {dest}')

# Also copy the updated summary CSV
if os.path.exists('outputs/summary_results.csv'):
    shutil.copy('outputs/summary_results.csv', os.path.join(OUT_DIR, 'summary_results.csv'))
    print(f'Copied outputs/summary_results.csv → {OUT_DIR}/summary_results.csv')

print('\nDone. Pull the phase4_dl_* folder + summary_results.csv back to your local repo and run notebooks/03_results.ipynb.')